# 금융분야 고객상담 데이터셋

In [ ]:
# ----------------------------------------------------------------
# 디렉토리 내 데이터 개수 확인용
# ----------------------------------------------------------------
from pathlib import Path

file_path = "../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)/Training/02.라벨링데이터/"
# 파일명 예) 21-1_bk_01_000029_001

def count_json_files(base_path: str):
    base = Path(base_path)
    total = 0

    for i in range(1,10):
        sub_dir = base/f"{i:02d}" # i를 두 자리 숫자로 표현하는 형식 지정
        # file_path 내부에 01-09까지의 디렉토리가 존재하기 때문에 이들 각각을 접근하기 위함

        if not sub_dir.exists():
            print(f"[{sub_dir.name}] 디렉토리 없음") # .name : 경로의 마지막 부분 이름 반환
            continue

        # glob : 특정 패턴에 맞는 파일이나 폴더를 찾아주는 기능
        #        > sub_dir 디렉토리 내 .json으로 끝나는 모든 파일을 찾음
        # 이를 리스트로 변환 후 개수 반환
        count = len(list(sub_dir.glob("*.json")))
        total += count
        print(f"[{sub_dir.name}] {count}개")

    print(f"\n총 json 파일 개수 : {total}개")

if __name__ == "__main__":
    count_json_files(file_path)

In [ ]:
# ----------------------------------------------------------------
# json 데이터 하나로 모아 df에 저장
# ----------------------------------------------------------------
import os
import json
import pandas as pd

# 데이터셋 경로
file_path = "../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)/Training/02.라벨링데이터/"

# 도메인, 중분류 코드 매핑
category_map = {
    "은행": {
        "01": "거래내역/잔액조회",
        "02": "중계요청/착오송금",
        "03": "자동이체조회",
        "04": "만기,연장/해지,수신",
        "05": "금융거래한도/비대면한도계좌",
        "06": "이자/연체금액",
        "07": "부수거래금리감면",
        "08": "대출문의(만기/연장/조회등)",
        "09": "환전문의",
    },

    "보험": {
        "01": "자동차보험상담",
        "02": "자동차사고접수",
        "03": "계약내용변경/해지",
        "04": "기타계약관련문의",
        "05": "보험금청구",
    },

    "증권": {
        "01": "HTS/MTS",
        "02": "계좌관리",
        "03": "신용거래/담보대출",
        "04": "자금이체/계좌제한",
        "05": "절세형금융상품",
        "06": "주식주문",
        "07": "증권계좌조회",
        "08": "해외주문",
    }
}

#파일명 분류코드 (도메인명으로 매핑하기 위함)
domain_map = {
    "bk" : "은행",
    "ins" : "보험",
    "sec" : "증권",
}

# json 내부 데이터 > df 형태로 변환
def flatten_json(data):
    """
    JSON 내부의 중첩된 dictionary를 펼쳐서
    DataFrame의 컬럼으로 사용할 수 있도록 변환
    """

    # --------------------------------
    # source
    # --------------------------------
    result = {}

    if "source" in data:
        source = data["source"]

        for key, value in source.items():
            result[f"source_{key}"] = value

    # --------------------------------
    # consulting
    # --------------------------------
    if "consulting" in data:
        consulting = data["consulting"]

        for key, value in consulting.items():
            result[f"consulting_{key}"] = value

    # --------------------------------
    # qa_data
    # --------------------------------
    if "qa_data" in data:

        # 하나의 JSON에 qa_data가 하나만 존재한다고 했으므로
        # 리스트의 첫 번째 데이터만 사용
        qa = data["qa_data"][0]

        for key, value in qa.items():

            # input은 dictionary이므로 한 번 더 펼침
            if key == "input" and isinstance(value, dict):

                for input_key, input_value in value.items():
                    result[f"qa_data_input_{input_key}"] = input_value

            else:
                result[f"qa_data_{key}"] = value

    return result

# 전체 json 파일 수집
all_data = []

# 01 ~ 08 디렉토리 순회
for folder_num in range(1, 10):

    folder_name = f"{folder_num:02d}"
    folder_path = os.path.join(file_path, folder_name)

    # 폴더가 존재하지 않는 경우 건너뜀
    if not os.path.exists(folder_path):
        print(f"[WARNING] 폴더 없음: {folder_path}")
        continue

    # 해당 폴더 내 JSON 파일 탐색
    for filename in os.listdir(folder_path):

        if not filename.lower().endswith(".json"):
            continue

        json_path = os.path.join(folder_path, filename)

        try:

            # -------------------------------
            # 파일명 분석
            # -------------------------------

            # 확장자 제거
            filename_without_ext = os.path.splitext(filename)[0]

            # 예:
            # 21-1_bk_01_000029_001
            parts = filename_without_ext.split("_")

            # 데이터분야별코드
            data_field_code = parts[0]

            # 분류코드
            domain_code = parts[1]

            # 중분류코드
            middle_code = parts[2]

            # 도메인명
            domain_name = domain_map.get(
                domain_code,
                f"Unknown({domain_code})"
            )

            # 중분류명
            middle_name = category_map.get(
                domain_name,
                {}
            ).get(
                middle_code,
                f"Unknown({middle_code})"
            )


            # -------------------------------
            # JSON 읽기
            # -------------------------------

            with open(
                json_path,
                "r",
                encoding="utf-8"
            ) as f:

                json_data = json.load(f)


            # JSON 내부 데이터 펼치기
            json_flattened = flatten_json(json_data)


            # -------------------------------
            # 메타데이터 + JSON 데이터 결합
            # -------------------------------

            row = {
                "파일명": filename,
                "데이터분야별코드": data_field_code,
                "분류코드": domain_code,
                "도메인명": domain_name,
                "중분류코드": middle_code,
                "중분류명": middle_name,
                "파일경로": json_path,
            }

            # JSON 내부 데이터 추가
            row.update(json_flattened)

            all_data.append(row)


        except Exception as e:

            print(f"[ERROR] {json_path}")
            print(f"       {e}")

# df 변환
df = pd.DataFrame(all_data)
df


In [ ]:
# ----------------------------------------------------------------
# 추출한 열 확인용
# ----------------------------------------------------------------
df.columns

In [ ]:
# ----------------------------------------------------------------
# 생성한 df 엑셀 파일로 저장
# ----------------------------------------------------------------
df.to_excel("output.xlsx", index=False)

In [ ]:
# ----------------------------------------------------------------
# 저장한 데이터셋 엑셀파일 불러오기
# ----------------------------------------------------------------
import pandas as pd

file_path = "../dataset/output.xlsx"

df = pd.read_excel(file_path)
df

In [ ]:
'''
Index(['파일명', '데이터분야별코드', '분류코드', '도메인명', '중분류코드', '중분류명', '파일경로',
       'source_source_institution', 'source_source_id', 'source_source_date',
       'source_client_gender', 'source_client_age', 'source_consulting_client',
       'source_consulting_client_type', 'source_source_length',
       'source_consulting_content', 'consulting_consulting_category',
       'consulting_consulting_topic', 'consulting_consulting_summary',
       'qa_data_qa_id', 'qa_data_task_category',
       'qa_data_consulting_situation', 'qa_data_qa_topic',
       'qa_data_consulting_purpose', 'qa_data_core_financial_terms',
       'qa_data_input_length', 'qa_data_instruction', 'qa_data_input_question',
       'qa_data_input_answer', 'qa_data_input_follow_up_question',
       'qa_data_output'],
      dtype='str')
'''